# New server endpoint

We chose to implement the alternative server endpoint presented in the lecture 12 that does the following by integrating 3 different models to do steps 2, 3, and 4:

```bash
(1) MoveNet -> (x_mn, y_mn)

(2) Cut leading and trailing frames

(3) MoveNet to Kinect 2d -> (x_k, y_k)

(4) Kinect 2d to Kinect 3d -> (x_k, y_k, z_k)
```

## Deployment

To deploy the service, we make sure that the models needed for cutting frames and transforming coordinates are accessible in our MLflow model registry such that our FastAPI server can download and use the models. We then created a new FastAPI endpoint function named get_csv_file (not the best name now that I think about it. See `ml/api/app/routers/datasets.py` in repository for details) that allows the user to request the server to take a given movenet csv file stored on the server and use the three models to generate a kinect file in csv. The kinect file is then sent to the user.

## Usage and user example

This section explains how the endpoint is used in combination with our web app to display a squat in 3D Kinect coordinates by providing a user example.

To use the endpoint, the user should start by navigating to our web app hosted at https://cu2170.camp.lnu.se. Make sure to be connected to the university network or the university VPN to access the hyperlink. Finally, navigate to the *Pose Recorder* tab.

![Pose Recorder tab](img/pose_recorder_tab.png)

Your browser should now display the interface as displayed below:

![Pose Recorder interface](img/pose_recorder_interface.png)

The first step is to generate MoveNet coordinates from a given squat video. The reader can do this by firstly activating the camera by clicking *Start Camera*. Remark that the web app must first load a model that estimates MoveNet keypoints on the target subject. The model is ready when the user can see some green lines and keypoints being displayed as shown below.

![Loading MoveNet](img/loading_movenet1.png)


You are now ready to record a video. Click *Record* to record a squat. When done, the user can click *Stop record* and then download the CSV file that stores the MoveNet keypoint coordinates by clicking *Save CSV*.

![Recording session](img/recording_session1.png)


To upload the generated CSV file to the server, click *Choose CSV* and choose the downloaded CSV file in your file system. Then simply click *Upload CSV* to store it on the server as shown below.

![Choosing CSV](img/choosing_csv.png)

![Saving CSV on server](img/saving_csv_on_server.png)

We are now ready to use the endpoint service. The uploaded CSV file is stored on the server, and we can now instruct the server to select the file and apply our three models to generate a cut 3D Kinect file that we can download. Below the *Available CSV* header is a dropdown menu where you can list and select an uploaded CSV file. Choose your recorded MoveNet file. Then toggle *Save CSV without non-squat frames* and click *Download Kinect CSV*. You should now have a downloaded CSV file that ends with `*_kinect_3d.csv`.

![Downloaded Kinect file](img/downloaded_kinect_file.png)

The final step is to display the squat. This is done in the *3D Skeleton Viewer* interface. Start by navigating to the interface by clicking the *3D Skeleton Viewer* tab.

![3D Skeleton Viewer tab](img/3d_skeletong_viewer_tab.png)

Your browser should now display the following interface:

![3D Skeleton Viewer interface](img/3d_skeleton_viewer_interface.png)

To display the squat, click *Choose 3D CSV* and choose the downloaded 3D Kinect CSV file. Then use the Frame slider to review the full squat as displayed.

![Rendering of squat](img/rendering_of_squat.png)

## Testing
Testing of the endpoint is mainly done manually by executing some of the steps in the previous section. Specifically, we tested that the server sent a cutted kinect file for a given uncut movenet file stored on the server.